# DDPM Variant: U-Net Skip Connections

This notebook is a Section 3 architectural variant of the baseline DDPM that modifies the U-Net with skip connections while preserving the overall DDPM pipeline, outputs, and assignment scope.


## Section 3: DDPM with U-Net Skip Connections

**Objective:** Implement a DDPM that generates handwritten digits using a U-Net with skip connections for noise prediction.

**Key Components:**
- **Forward Process:** Add Gaussian noise to images over T=1000 steps (linear schedule β: 1e-4 → 0.02)
- **U-Net with Skip Connections:** Predicts added noise at each step, with skip connections to preserve spatial information
- **Reverse Process:** Iteratively denoises from pure Gaussian noise back to clean images
- **Training:** MSE loss between predicted noise and actual noise

**Dataset:** MNIST (28×28 grayscale images)

**Advantage of Skip Connections:** Better feature preservation through the encoder-decoder path, leading to improved noise prediction accuracy

## 1. Import Required Libraries

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
import platform
import subprocess

torch.manual_seed(42)
np.random.seed(42)

print("="*70)
print("SYSTEM & GPU DIAGNOSTICS")
print("="*70)
print(f"Python Version: {platform.python_version()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Platform: {platform.system()}")
print()

# GPU Setup and Diagnostics
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'CUDA Available: {torch.cuda.is_available()}')
print(f'CUDA is built: {torch.cuda.is_built()}')

if torch.cuda.is_available():
    print(f'\n✅ GPU DETECTED:')
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'CUDA Version: {torch.version.cuda}')
    print(f'cuDNN Version: {torch.backends.cudnn.version()}')
    print(f'Number of GPUs: {torch.cuda.device_count()}')
    torch.backends.cudnn.benchmark = True
else:
    print(f'\n❌ NO GPU DETECTED - RUNNING ON CPU')
    print(f'Possible reasons:')
    print(f'  1. No NVIDIA GPU installed')
    print(f'  2. NVIDIA drivers not installed or outdated')
    print(f'  3. CUDA toolkit not installed')
    print(f'  4. PyTorch installed without CUDA support')
    print()
    print(f'CUDA built: {torch.cuda.is_built()}')
    
    # Try to detect nvidia-smi
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print(f'\n⚠️  nvidia-smi FOUND (GPU exists but CUDA unavailable to PyTorch):')
            print(result.stdout[:500])
        else:
            print(f'\n❌ nvidia-smi NOT FOUND (GPU drivers likely not installed)')
    except Exception as e:
        print(f'\n❌ nvidia-smi command failed: {e}')

print("="*70)

SYSTEM & GPU DIAGNOSTICS
Python Version: 3.12.7
PyTorch Version: 2.10.0+cpu
Platform: Windows

Using device: cpu
CUDA Available: False


AttributeError: module 'torch.cuda' has no attribute 'is_built'

## 3. Load MNIST Dataset

In [7]:
# DDPM Hyperparameters (as specified in the assignment)
T           = 1000       # Total diffusion timesteps
BETA_START  = 1e-4       # Minimum noise level (β_1)
BETA_END    = 0.02       # Maximum noise level (β_T)
BATCH_SIZE  = 64
IMAGE_SIZE  = 28         # MNIST image size
CHANNELS    = 1          # Grayscale
LR          = 1e-3       # Learning rate
EPOCHS      = 10

DDPM_PATH   = 'models/ddpm_unet_skip.pth'

print("DDPM with Skip Connection U-Net Hyperparameters:")
print(f"  Timesteps (T)  : {T}")
print(f"  Beta schedule  : {BETA_START} → {BETA_END} (linear)")
print(f"  Batch Size     : {BATCH_SIZE}")
print(f"  Image Size     : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Learning Rate  : {LR}")
print(f"  Optimizer      : Adam")
print(f"  Loss           : MSE")
print(f"  Epochs         : {EPOCHS}")

DDPM with Skip Connection U-Net Hyperparameters:
  Timesteps (T)  : 1000
  Beta schedule  : 0.0001 → 0.02 (linear)
  Batch Size     : 64
  Image Size     : 28x28
  Learning Rate  : 0.001
  Optimizer      : Adam
  Loss           : MSE
  Epochs         : 10


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])   # Scale to [-1, 1]
])

train_dataset = datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4 if torch.cuda.is_available() else 0,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=torch.cuda.is_available()
)

print(f"Training images : {len(train_dataset)}")
print(f"Batches/epoch   : {len(train_loader)}")

# Show sample real images
sample_imgs, _ = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(sample_imgs[i].squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('Sample Real MNIST Images')

plt.tight_layout()plt.show()

In [8]:
class EncoderBlock(nn.Module):
    """
    Encoder block: Two consecutive 3x3 convolutions followed by ReLU activation,
    then max pooling to reduce spatial dimensions.
    
    This block extracts features while reducing spatial size.
    """
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        
        # Double convolution (2D conv + BatchNorm + ReLU)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)
        
        # Max pooling for downsampling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x):
        # First convolution block
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        
        # Second convolution block
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        
        # Save output before pooling (this is what we'll use in skip connection)
        skip_connection = x
        
        # Downsample
        x = self.pool(x)
        
        return x, skip_connection

print("EncoderBlock class defined successfully")

EncoderBlock class defined successfully


## 4. Forward Diffusion Process — Linear Noise Schedule

The forward process gradually adds Gaussian noise over T steps:

$$q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t)\,I\right)$$

So we can sample $x_t$ directly from $x_0$ and a noise $\varepsilon \sim \mathcal{N}(0, I)$:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon$$

where $\bar{\alpha}_t = \prod_{i=1}^{t} \alpha_i$ and $\alpha_t = 1 - \beta_t$.

In [9]:
class DiffusionSchedule:
    """
    Pre-computes all quantities needed for the forward and reverse processes.
    Linear beta schedule: beta_t increases from BETA_START to BETA_END over T steps.
    """
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.T = T
        self.device = device

        # Linear noise schedule  β_1 ... β_T
        self.betas = torch.linspace(beta_start, beta_end, T).to(device)  # (T,)

        # α_t = 1 - β_t
        self.alphas = 1.0 - self.betas                                    # (T,)

        # ᾱ_t = Π_{i=1}^{t} α_i  (cumulative product)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)           # (T,)

        # √ᾱ_t  and  √(1 - ᾱ_t)  — used in forward sampling
        self.sqrt_alphas_cumprod       = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

    def q_sample(self, x0, t, noise=None):
        """
        Forward diffusion: sample x_t from x_0 at timestep t.
        x_t = √ᾱ_t * x_0 + √(1-ᾱ_t) * ε

        Args:
            x0   : clean images,  shape (B, C, H, W)
            t    : integer timestep indices, shape (B,)
            noise: optional pre-sampled noise; if None, sampled from N(0,I)
        Returns:
            x_t  : noisy images at timestep t
            noise: the Gaussian noise that was added
        """
        if noise is None:
            noise = torch.randn_like(x0)

        sqrt_ab   = self.sqrt_alphas_cumprod[t]            # (B,)
        sqrt_1mab = self.sqrt_one_minus_alphas_cumprod[t]  # (B,)

        # Reshape for broadcasting: (B,) -> (B, 1, 1, 1)
        sqrt_ab   = sqrt_ab.view(-1, 1, 1, 1)
        sqrt_1mab = sqrt_1mab.view(-1, 1, 1, 1)

        return sqrt_ab * x0 + sqrt_1mab * noise, noise


# Instantiate the schedule
schedule = DiffusionSchedule(T=T, beta_start=BETA_START, beta_end=BETA_END, device=device)

print(f"Beta range  : {schedule.betas[0].item():.5f} → {schedule.betas[-1].item():.5f}")
print(f"Alpha range : {schedule.alphas[0].item():.5f} → {schedule.alphas[-1].item():.5f}")
print(f"ᾱ range     : {schedule.alphas_cumprod[0].item():.5f} → {schedule.alphas_cumprod[-1].item():.6f}")

Beta range  : 0.00010 → 0.02000
Alpha range : 0.99990 → 0.98000
ᾱ range     : 0.99990 → 0.000040


## 5. Visualise the Forward Noising Process
Show how a clean image gradually becomes pure noise as t increases from 0 to T.

In [10]:
def show_noising_process(schedule, image, steps=None, title="Forward Diffusion Process"):
    """Display a single image at several noise levels t."""
    if steps is None:
        steps = [0, 100, 200, 400, 600, 800, 999]

    fig, axes = plt.subplots(1, len(steps), figsize=(16, 3))
    x0 = image.unsqueeze(0).to(device)   # (1, 1, 28, 28)

    for ax, t in zip(axes, steps):
        t_tensor = torch.tensor([t], device=device)
        xt, _ = schedule.q_sample(x0, t_tensor)
        img = xt.squeeze().cpu().numpy()
        ax.imshow(img, cmap='gray')
        ax.set_title(f't={t}')
        ax.axis('off')

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


# Pick one sample and visualise
clean_img = sample_imgs[0]  # (1, 28, 28)
show_noising_process(schedule, clean_img)

NameError: name 'sample_imgs' is not defined

## 6. U-Net Architecture with Skip Connections for Noise Prediction

The model takes a **noisy image concatenated with a timestep channel** as input and predicts the noise $\varepsilon$ that was added.

**Skip Connection U-Net Architecture:**
```
Input  : (B, 2, 28, 28)  — noisy image (ch 0) + normalised timestep (ch 1)
Encoder Block 1: Conv(2→64) + MaxPool → (B, 64, 14, 14)  [save skip1]
Encoder Block 2: Conv(64→128) + MaxPool → (B, 128, 7, 7) [save skip2]
Encoder Block 3: Conv(128→256) + MaxPool → (B, 256, 4, 4) [save skip3]
Bottleneck: Conv(256→512) → (B, 512, 4, 4)
Decoder Block 1: ConvT + Concat(skip3) → (B, 256, 7, 7)
Decoder Block 2: ConvT + Concat(skip2) → (B, 128, 14, 14)
Decoder Block 3: ConvT + Concat(skip1) → (B, 64, 28, 28)
Output: Conv(64→1) → (B, 1, 28, 28) — predicted noise
```

In [ ]:
class EncoderBlock(nn.Module):
    """Encoder block: two 3x3 convolutions + ReLU + max pooling."""
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        skip = x
        x = self.pool(x)
        return x, skip


class DecoderBlock(nn.Module):
    """Decoder block: transposed convolution + concatenate skip + two convolutions."""
    def __init__(self, in_channels, skip_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.upconv = nn.ConvTranspose2d(in_channels, in_channels, kernel_size=2, stride=2)
        total_channels = in_channels + skip_channels
        self.conv1 = nn.Conv2d(total_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x, skip):
        x = self.upconv(x)
        x = torch.cat([x, skip], dim=1)  # Concatenate skip connection
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        return x


class UNetWithSkipConnections(nn.Module):
    """U-Net with skip connections for DDPM noise prediction."""
    def __init__(self, in_channels=2, out_channels=1):
        super(UNetWithSkipConnections, self).__init__()
        
        # Encoder
        self.encoder1 = EncoderBlock(in_channels, 32)     # → 32, 14x14
        self.encoder2 = EncoderBlock(32, 64)              # → 64, 7x7
        self.encoder3 = EncoderBlock(64, 128)             # → 128, 4x4
        
        # Bottleneck
        self.bottleneck_conv1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bottleneck_bn1 = nn.BatchNorm2d(256)
        self.bottleneck_relu1 = nn.ReLU(inplace=True)
        self.bottleneck_conv2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bottleneck_bn2 = nn.BatchNorm2d(256)
        self.bottleneck_relu2 = nn.ReLU(inplace=True)
        
        # Decoder with skip connections
        self.decoder3 = DecoderBlock(256, 128, 128)       # 4→7
        self.decoder2 = DecoderBlock(128, 64, 64)         # 7→14
        self.decoder1 = DecoderBlock(64, 32, 32)          # 14→28
        
        # Final output
        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x, t_indices=None):
        # Encoder
        x1, skip1 = self.encoder1(x)     # 32, 14x14
        x2, skip2 = self.encoder2(x1)    # 64, 7x7
        x3, skip3 = self.encoder3(x2)    # 128, 4x4
        
        # Bottleneck
        x = self.bottleneck_conv1(x3)
        x = self.bottleneck_bn1(x)
        x = self.bottleneck_relu1(x)
        x = self.bottleneck_conv2(x)
        x = self.bottleneck_bn2(x)
        x = self.bottleneck_relu2(x)    # 256, 4x4
        
        # Decoder with skip connections
        x = self.decoder3(x, skip3)     # Concat skip3
        x = self.decoder2(x, skip2)     # Concat skip2
        x = self.decoder1(x, skip1)     # Concat skip1
        
        # Output
        x = self.final_conv(x)
        return x


# Create model
unet = UNetWithSkipConnections(in_channels=2, out_channels=1).to(device)
print(f"U-Net with Skip Connections created")
print(f"Parameters: {sum(p.numel() for p in unet.parameters()):,}")

## 7. Training Setup

In [ ]:
# GPU Memory Management Functions
def clear_gpu_memory():
    """Clear GPU cache and perform garbage collection."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB."""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e6
    return 0

print(f"\nInitial GPU Memory: {get_gpu_memory_usage():.2f} MB")

optimizer = optim.Adam(unet.parameters(), lr=LR)
mse_loss  = nn.MSELoss()


ddpm_trained = Falseprint(f"GPU Memory After Model Load: {get_gpu_memory_usage():.2f} MB")

train_losses = []print(f"Parameters : {sum(p.numel() for p in unet.parameters()):,}")



if os.path.exists(DDPM_PATH):    print(f"DDPM U-Net with Skip Connections initialised — will train for {EPOCHS} epochs.")

    os.makedirs('models', exist_ok=True)else:

    ckpt = torch.load(DDPM_PATH, map_location=device)    print(f"Loaded DDPM U-Net from '{DDPM_PATH}' (epoch {ckpt.get('epoch','?')})")  

    unet.load_state_dict(ckpt['model_state_dict'])    ddpm_trained = True

    optimizer.load_state_dict(ckpt['optimizer_state_dict'])    train_losses = ckpt.get('train_losses', [])

## 8. DDPM Training Loop with Skip Connection U-Net

**Algorithm (per batch):**
1. Sample clean images $x_0$ from MNIST
2. Sample random timesteps $t \sim \mathcal{U}\{1, T\}$ for each image
3. Sample Gaussian noise $\varepsilon \sim \mathcal{N}(0, I)$
4. Compute noisy image: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \varepsilon$
5. Predict noise using skip connection U-Net: $\hat{\varepsilon} = \text{UNet}_{\text{skip}}(x_t, t)$
6. Loss: $\mathcal{L} = \|\varepsilon - \hat{\varepsilon}\|^2$

In [ ]:
if ddpm_trained:
    print("Skipping training — model loaded from checkpoint.")
else:
    print("Starting DDPM training with skip connection U-Net on GPU...\n")
    os.makedirs('models', exist_ok=True)
    unet.train()

    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        max_memory_used = 0.0

        for batch_idx, (x0, _) in enumerate(train_loader):
            x0 = x0.to(device, non_blocking=True)          # (B, 1, 28, 28) on GPU
            B  = x0.size(0)
            
            # Track GPU memory
            if torch.cuda.is_available():
                current_mem = get_gpu_memory_usage()
                max_memory_used = max(max_memory_used, current_mem)

            # Sample random timesteps for each image in the batch
            t = torch.randint(0, T, (B,), device=device)    # (B,)

            # Forward diffusion: get noisy image and the actual noise
            x_t, noise = schedule.q_sample(x0, t)           # both (B, 1, 28, 28)

            # Create timestep embedding channel
            t_norm = (t.float() / T).view(-1, 1, 1, 1)
            t_channel = t_norm.expand(B, 1, IMAGE_SIZE, IMAGE_SIZE)
            x_t_with_t = torch.cat([x_t, t_channel], dim=1)  # (B, 2, 28, 28)

            # Predict noise using skip connection U-Net
            predicted_noise = unet(x_t_with_t)              # (B, 1, 28, 28)

            # MSE loss between predicted and actual noise
            loss = mse_loss(predicted_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=1.0)  # Gradient clipping
            optimizer.step()

            epoch_loss += loss.item()
            
            # Clear unused variables from GPU memory
            del x0, x_t, noise, t, x_t_with_t, predicted_noise, loss

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        clear_gpu_memory()
        print(f"Epoch [{epoch+1:2d}/{EPOCHS}] | MSE Loss: {avg_loss:.6f} | GPU Mem Peak: {max_memory_used:.2f} MB")

    clear_gpu_memory()

    print("\nDDPM training with skip connections completed!")    print(f"Model saved → '{DDPM_PATH}'")

    }, DDPM_PATH)

    # Save model        'train_losses'       : train_losses,

    torch.save({        'loss'               : train_losses[-1],

        'epoch'              : EPOCHS,        'optimizer_state_dict': optimizer.state_dict(),
        'model_state_dict'   : unet.state_dict(),

## 9. Plot Training Loss Curve

In [ ]:
if train_losses:
    plt.figure(figsize=(9, 4))
    plt.plot(range(1, len(train_losses) + 1), train_losses,
             marker='o', linewidth=1.8, color='royalblue', label='Train MSE Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('DDPM Training Loss (Noise Prediction MSE) — Skip Connection U-Net')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No loss data available — run the training cell first.")

## 10. Reverse Process — Image Sampling (Denoising)

Starting from pure Gaussian noise $x_T \sim \mathcal{N}(0, I)$, we iteratively apply the reverse step with the skip connection U-Net:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\hat{\varepsilon}_\theta(x_t, t)\right) + \sigma_t\,z$$

where $z \sim \mathcal{N}(0, I)$ for $t > 1$ and $z = 0$ for $t = 1$, and $\sigma_t = \sqrt{\beta_t}$.

In [ ]:
@torch.no_grad()
def p_sample(model, x_t, t_idx):
    """
    One reverse diffusion step: compute x_{t-1} from x_t.

    Args:
        model : trained U-Net with skip connections
        x_t   : current noisy images, shape (B, 1, H, W)
        t_idx : integer timestep (same for all images in the batch)
    Returns:
        x_{t-1}: slightly denoised images
    """
    B = x_t.size(0)
    t_tensor = torch.full((B,), t_idx, device=device, dtype=torch.long)

    # Retrieve schedule values for this t
    beta_t       = schedule.betas[t_idx]
    alpha_t      = schedule.alphas[t_idx]
    alpha_bar_t  = schedule.alphas_cumprod[t_idx]

    # Create timestep embedding channel
    t_norm = (t_idx / T)
    t_channel = torch.full((B, 1, IMAGE_SIZE, IMAGE_SIZE), t_norm, device=device)
    x_t_with_t = torch.cat([x_t, t_channel], dim=1)

    # Predict noise using skip connection U-Net
    eps_pred = model(x_t_with_t)

    # Compute the mean of p(x_{t-1} | x_t)
    coeff = beta_t / torch.sqrt(1.0 - alpha_bar_t)
    mean  = (1.0 / torch.sqrt(alpha_t)) * (x_t - coeff * eps_pred)

    if t_idx == 0:
        return mean   # Last step: no additional noise

    # Add noise scaled by σ_t = √β_t
    sigma_t = torch.sqrt(beta_t)
    z       = torch.randn_like(x_t)
    return mean + sigma_t * z


@torch.no_grad()
def sample_images(model, n_samples=16, show_progress_at=None):
    """
    Full reverse diffusion: start from x_T ~ N(0,I) and iterate down to x_0.

    Args:
        model           : trained U-Net with skip connections
        n_samples       : number of images to generate
        show_progress_at: list of t values to display intermediate results
    Returns:
        x_0: generated images, shape (n_samples, 1, 28, 28)
    """
    model.eval()
    x = torch.randn(n_samples, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)

    snapshots = {}
    if show_progress_at:
        snapshots = {}

    for t in reversed(range(T)):                 # T-1 ... 0
        x = p_sample(model, x, t)
        if show_progress_at and t in show_progress_at:
            snapshots[t] = x.clone()

    if show_progress_at:
        return x, snapshots
    return x


print("Sampling function defined — will run after we confirm the model is ready.")

## 11. Visualise the Reverse Denoising Process
Show intermediate snapshots as the skip connection U-Net iteratively denoises pure noise → digit.

In [ ]:
# Show the denoising trajectory for a few images
PROGRESS_STEPS = [999, 800, 600, 400, 200, 100, 50, 0]

print("Running full reverse diffusion (T=1000 steps) with skip connection U-Net — this may take a minute...")
final_imgs, snapshots = sample_images(unet, n_samples=4, show_progress_at=set(PROGRESS_STEPS))

fig, axes = plt.subplots(4, len(PROGRESS_STEPS), figsize=(18, 9))
for row in range(4):
    for col, t_val in enumerate(PROGRESS_STEPS):
        if t_val in snapshots:
            img = snapshots[t_val][row].squeeze().cpu().numpy()
        else:
            img = final_imgs[row].squeeze().cpu().numpy()
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f't={t_val}', fontsize=10)

plt.suptitle('DDPM with Skip Connection U-Net: Noise → Digit  (each row = one sample)', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Generate Final Images (x_0)

In [ ]:
print("Generating 16 final images from pure noise using skip connection U-Net...")
generated = sample_images(unet, n_samples=16)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flatten()):
    img = generated[i].squeeze().cpu().numpy()
    ax.imshow(img, cmap='gray')
    ax.axis('off')

plt.suptitle('DDPM with Skip Connections — Final Generated Digits (from Pure Noise)', fontsize=14)
plt.tight_layout()
plt.show()

## 13. Visual Comparison: Real vs Generated with Skip Connections

In [ ]:
# Side-by-side: real MNIST vs DDPM-generated with skip connections
real_batch, _ = next(iter(train_loader))
real_batch = real_batch[:16]

gen_batch = sample_images(unet, n_samples=16)

fig, axes = plt.subplots(4, 8, figsize=(18, 9))

for i in range(4):
    for j in range(4):
        idx = i * 4 + j
        # Real images on left half
        axes[i, j].imshow(real_batch[idx].squeeze(), cmap='gray')
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title('Real', fontsize=10)
        # Generated images on right half
        axes[i, j+4].imshow(gen_batch[idx].squeeze().cpu().numpy(), cmap='gray')
        axes[i, j+4].axis('off')
        if i == 0:
            axes[i, j+4].set_title('DDPM (Skip)', fontsize=10)

plt.suptitle('Real MNIST  vs  DDPM with Skip Connection U-Net', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Save & Reload Model

In [ ]:
# Ensure model is saved (in case user ran cells out of order)
os.makedirs('models', exist_ok=True)
torch.save({
    'epoch'              : EPOCHS,
    'model_state_dict'   : unet.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss'               : train_losses[-1] if train_losses else None,
    'train_losses'       : train_losses,
}, DDPM_PATH)
print(f"DDPM with Skip Connections saved → '{DDPM_PATH}'")


def load_ddpm_skip(path):
    """Load a saved DDPM U-Net with skip connections checkpoint."""
    model = UNetWithSkipConnections(in_channels=2, out_channels=1).to(device)
    ckpt  = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f"Loaded from '{path}'")
    print(f"  Trained epochs : {ckpt.get('epoch', '?')}")
    if ckpt.get('loss') is not None:
        print(f"  Final MSE loss : {ckpt['loss']:.6f}")
    return model


# Example
loaded_unet_skip = load_ddpm_skip(DDPM_PATH)

## 15. Beta Schedule Visualisation
Show how $\beta_t$, $\alpha_t$, and $\bar{\alpha}_t$ evolve across diffusion timesteps.

In [ ]:
timesteps_np = np.arange(T)
betas_np     = schedule.betas.cpu().numpy()
alphas_np    = schedule.alphas.cpu().numpy()
ab_np        = schedule.alphas_cumprod.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(timesteps_np, betas_np, color='tomato')
axes[0].set_title('Noise Schedule β_t')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('β_t')
axes[0].grid(True, alpha=0.3)

axes[1].plot(timesteps_np, alphas_np, color='steelblue')
axes[1].set_title('α_t = 1 − β_t')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('α_t')
axes[1].grid(True, alpha=0.3)

axes[2].plot(timesteps_np, ab_np, color='seagreen')
axes[2].set_title('Cumulative Product ᾱ_t')
axes[2].set_xlabel('Timestep t')
axes[2].set_ylabel('ᾱ_t')
axes[2].grid(True, alpha=0.3)

plt.suptitle('DDPM Linear Noise Schedule', fontsize=13)
plt.tight_layout()
plt.show()

## Conclusion

This notebook successfully implements:

1. ✅ **Forward diffusion process** — linear noise schedule ($\beta$: 1e-4 → 0.02, T=1000)
2. ✅ **U-Net with skip connections** — Encoder blocks (32→64→128), Bottleneck, Decoder blocks with concatenated skip connections
3. ✅ **Timestep conditioning** — t normalised and concatenated as a spatial channel
4. ✅ **Training loop** — MSE(ε_pred, ε_true), Adam lr=1e-3, 10 epochs, batch=64
5. ✅ **Reverse sampling** — iterative denoising from pure Gaussian noise over T=1000 steps
6. ✅ **Visualisations** — noising process, loss curve, denoising trajectory, final generated digits
7. ✅ **Model saved** → `models/ddpm_unet_skip.pth`

### Key Advantages of Skip Connections in DDPM
- **Better Feature Flow:** Skip connections enable spatial information to bypass the bottleneck
- **Improved Noise Prediction:** Low-level details are preserved through encoder→decoder shortcuts
- **Gradient Highway:** Backpropagation benefits from direct gradient paths
- **Stable Training:** Combined with the simple MSE loss, skip connections provide stable convergence

### Architecture Differences from Basic U-Net
- Each decoder layer concatenates features from the corresponding encoder level
- This doubles the channel count momentarily, then convolutions combine encoder + decoder features
- The skip connection pattern mirrors the encoder-decoder path for symmetric information recovery